# Lab 4: Exploratory Data Analysis with SQL

The SpaceX launch table is loaded into a SQLite database and queried with SQL.

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
os.makedirs('results', exist_ok=True)

def save_result(name, value):
    path = 'results/results.json'
    data = json.load(open(path)) if os.path.exists(path) else {}
    data[name] = value
    json.dump(data, open(path, 'w'), indent=2, default=str)

import sqlite3
import pandas as pd
con = sqlite3.connect('my_data1.db')
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv')
df.to_sql('SPACEXTBL', con, if_exists='replace', index=False, method='multi')

def run(query):
    return pd.read_sql(query, con)

Remove blank rows from the table:

In [2]:
con.execute('DROP TABLE IF EXISTS SPACEXTABLE;')
con.execute('CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL;')
run('SELECT COUNT(*) AS rows FROM SPACEXTABLE;')

,rows
0,101


### Task 1: Names of the unique launch sites

In [3]:
query = """
SELECT DISTINCT Launch_Site FROM SPACEXTABLE;
"""
res = run(query)
save_result('sql_unique_sites', res.to_dict(orient='records'))
res

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### Task 2: 5 records where launch sites begin with 'CCA'

In [4]:
query = """
SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5;
"""
res = run(query)
save_result('sql_cca5', res.to_dict(orient='records'))
res

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3: Total payload mass carried by boosters launched by NASA (CRS)

In [5]:
query = """
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)';
"""
res = run(query)
save_result('sql_nasa_crs_total', res.to_dict(orient='records'))
res

,Total_Payload_Mass
0,45596


### Task 4: Average payload mass carried by booster version F9 v1.1

In [6]:
query = """
SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass FROM SPACEXTABLE WHERE Booster_Version = 'F9 v1.1';
"""
res = run(query)
save_result('sql_f9v11_avg', res.to_dict(orient='records'))
res

,Avg_Payload_Mass
0,2928.4


### Task 5: Date of the first successful landing on a ground pad

In [7]:
query = """
SELECT MIN(Date) AS First_Ground_Pad_Success FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)';
"""
res = run(query)
save_result('sql_first_ground_pad', res.to_dict(orient='records'))
res

,First_Ground_Pad_Success
0,2015-12-22


### Task 6: Boosters with drone ship success and payload mass between 4000 and 6000 kg

In [8]:
query = """
SELECT Booster_Version FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000;
"""
res = run(query)
save_result('sql_drone_4000_6000', res.to_dict(orient='records'))
res

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


### Task 7: Total number of successful and failed mission outcomes

In [9]:
query = """
SELECT TRIM(Mission_Outcome) AS Mission_Outcome, COUNT(*) AS Total
FROM SPACEXTABLE GROUP BY TRIM(Mission_Outcome);
"""
res = run(query)
save_result('sql_mission_outcomes', res.to_dict(orient='records'))
res

,Mission_Outcome,Total
0,Failure (in flight),1
1,Success,99
2,Success (payload status unclear),1


### Task 8: Booster versions that carried the maximum payload mass (subquery)

In [10]:
query = """
SELECT DISTINCT Booster_Version FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE);
"""
res = run(query)
save_result('sql_max_payload_boosters', res.to_dict(orient='records'))
res

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


### Task 9: 2015 records with failed drone ship landings: month, booster version, launch site

In [11]:
query = """
SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site
FROM SPACEXTABLE
WHERE substr(Date, 1, 4) = '2015' AND Landing_Outcome = 'Failure (drone ship)';
"""
res = run(query)
save_result('sql_failed_drone_2015', res.to_dict(orient='records'))
res

,Month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Task 10: Rank landing outcomes between 2010-06-04 and 2017-03-20 (descending)

In [12]:
query = """
SELECT Landing_Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC;
"""
res = run(query)
save_result('sql_rank_outcomes', res.to_dict(orient='records'))
res

,Landing_Outcome,Outcome_Count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1


In [13]:
con.close()